# Compare Pipeline Cutouts vs H5 Ground Truth

For a selected lens system, compare:
- Pipeline cutouts (visit image and difference image from step2)
- Original H5 stamp data, rotated by the visit rotation angle

This verifies that the injection + subtraction pipeline produces results consistent with the input H5 data.

In [ ]:
import numpy as np
import h5py
import json
import re
import matplotlib.pyplot as plt
from astropy.io import fits
from astropy.table import Table
from astropy.wcs import WCS as astropyWCS
from scipy.ndimage import rotate as scipy_rotate
from scipy.ndimage import gaussian_filter

In [ ]:
%matplotlib inline

## Configuration

In [ ]:
# Path to pipeline output folder
OUTPUT_FOLDER = "/home/sfu/shared_lagn_injection/v3.1_ecdfs_u_1"

# Path to merged cleaned H5 file
H5_FILE = "/home/sfu/shared_lagn_injection/lens_finding_postage_stamps_dataset/3000sqdeg_lsst_1y_sample_merged_cleaned.h5"

# Which stamp to inspect (injection_id, 0-indexed)
STAMP_INDEX = 0

# Constants
PIX2ARCSEC = 0.2
STAMP_CENTER = 16  # center pixel of 33x33 stamp
FWHM_ARCSEC = 1.0  # approximate PSF FWHM
PSF_SIGMA = FWHM_ARCSEC / (2.355 * PIX2ARCSEC)  # ~2.12 pixels

MAG_ZPS_CPS = {
    'u': 26.52, 'g': 28.51, 'r': 28.36,
    'i': 28.17, 'z': 27.78, 'y': 26.82,
}

def compute_coord_after_rot(x_c, y_c, delta_x, delta_y, rotation_angle):
    """Rotate (delta_x, delta_y) clockwise by rotation_angle (deg), add to (x_c, y_c)."""
    theta = np.deg2rad(rotation_angle)
    dx_new = delta_x * np.cos(theta) + delta_y * np.sin(theta)
    dy_new = -delta_x * np.sin(theta) + delta_y * np.cos(theta)
    return x_c + dx_new, y_c + dy_new

print(f"PSF_SIGMA: {PSF_SIGMA:.2f} pixels")

## Load metadata and injection catalog

In [ ]:
# Load run metadata from numbers_log.jsonl
# Each line is a JSON record; pick the one matching iter_index from folder name
iter_index = int(OUTPUT_FOLDER.rstrip('/').split('_')[-1])

jsonl_file = f"../v3.1_u/numbers_log.jsonl"
with open(jsonl_file) as f:
    for line in f:
        rec = json.loads(line.strip())
        if rec.get('iter_index') == iter_index:
            break

visit = rec['visit']
detector = rec['detector']
tract = rec['tract']
patch = rec['patch']
band = rec['band']
visit_rotation_angle = rec['visit_rotation_angle']

print(f"visit={visit}, detector={detector}, tract={tract}, patch={patch}")
print(f"band={band}, rotation_angle={visit_rotation_angle:.2f} deg")

In [ ]:
# Build file paths
tag = f"{visit}_{detector}_{tract}_{patch}_{band}"
cutout_visit_file = f"{OUTPUT_FOLDER}/cutout/visit_{tag}_image.fits"
cutout_diff_file = f"{OUTPUT_FOLDER}/cutout/diff_{tag}_image.fits"
inj_catalog_file = f"{OUTPUT_FOLDER}/inj_catalog_visit_{visit}_{detector}.fits"

print(f"Catalog: {inj_catalog_file}")
print(f"Visit cutout: {cutout_visit_file}")
print(f"Diff cutout: {cutout_diff_file}")

In [ ]:
# Load injection catalog
inj_catalog = Table.read(inj_catalog_file)
stamps = inj_catalog[inj_catalog['source_type'] == 'Stamp']
points = inj_catalog[inj_catalog['source_type'] == 'Star']
print(f"Total entries: {len(inj_catalog)}, stamps: {len(stamps)}, points: {len(points)}")

# Select target stamp
target_stamp = stamps[STAMP_INDEX]
assert target_stamp['injection_id'] == STAMP_INDEX, \
    f"injection_id mismatch: {target_stamp['injection_id']} != {STAMP_INDEX}"
print(f"\nTarget stamp: injection_id={target_stamp['injection_id']}, "
      f"RA={target_stamp['ra']:.6f}, DEC={target_stamp['dec']:.6f}, "
      f"mag={target_stamp['mag']:.2f}")

# Extract system_index from stamp filename
stamp_path = target_stamp['stamp']
m = re.search(r'system_(\d+)_static', stamp_path)
system_index = int(m.group(1))
print(f"system_index: {system_index}")

# Find associated point sources by RA/DEC proximity
stamp_ra, stamp_dec = target_stamp['ra'], target_stamp['dec']
sep_arcsec = np.sqrt(
    ((points['ra'] - stamp_ra) * np.cos(np.deg2rad(stamp_dec)) * 3600)**2 +
    ((points['dec'] - stamp_dec) * 3600)**2
)
associated_points = points[sep_arcsec < 5.0]

time_index = int(associated_points['time_index'][0])
print(f"time_index: {time_index}")
print(f"Associated point sources: {len(associated_points)}")
for p in associated_points:
    print(f"  id={p['injection_id']}, mag={p['mag']:.2f}")

## Load pipeline cutouts

In [ ]:
ext_name = f"ID_{STAMP_INDEX:04d}"

with fits.open(cutout_visit_file) as hdul:
    cutout_visit = hdul[ext_name].data
    cutout_visit_header = hdul[ext_name].header

with fits.open(cutout_diff_file) as hdul:
    cutout_diff = hdul[ext_name].data
    cutout_diff_header = hdul[ext_name].header

print(f"Cutout visit shape: {cutout_visit.shape}")
print(f"Cutout diff shape: {cutout_diff.shape}")
print(f"INJ_ID from header: {cutout_visit_header['INJ_ID']}")

## Load H5 data

In [ ]:
def mag2flux_cps(mag, band):
    """Convert magnitude to flux in counts per second using CPS zero points."""
    return 10**((MAG_ZPS_CPS[band] - mag) / 2.5)

with h5py.File(H5_FILE, 'r') as hf:
    group = hf[f'lsst_lens_{system_index}']

    # Static image (host galaxy + lensed arc)
    h5_static = group['static_image'][band]['lens_plus_lensed_agn_host'][:]

    # All observations (to compute mean stamp for template comparison)
    all_obs = group['postage_stamps'][band]['all_observations'][:]
    h5_stamp_mean = np.mean(all_obs, axis=0)

    # Epoch stamp (static + point sources at this epoch)
    h5_stamp = all_obs[time_index]

    # Point source light curve mags and mean fluxes
    light_curves = group['light_curves']
    n_images = len(light_curves)
    point_mags_h5 = []
    point_mean_flux_h5 = []
    for img_ind in range(n_images):
        lc = light_curves[f'image_{img_ind}'][band][:]
        point_mags_h5.append(float(lc[time_index]))
        # Mean flux (not mean mag), consistent with template injection in lib/inject.py
        fluxes = mag2flux_cps(lc, band)
        point_mean_flux_h5.append(float(np.mean(fluxes)))

    # Point source pixel offsets (unrotated)
    metadata = group['metadata']
    point_dx, point_dy = [], []
    for img_ind in range(n_images):
        dx = metadata.attrs[f'point_source_light_{band}_ra_image_{img_ind}'] / PIX2ARCSEC
        dy = metadata.attrs[f'point_source_light_{band}_dec_image_{img_ind}'] / PIX2ARCSEC
        point_dx.append(dx)
        point_dy.append(dy)

print(f"H5 static shape: {h5_static.shape}")
print(f"H5 stamp shape: {h5_stamp.shape}")
print(f"H5 stamp mean shape: {h5_stamp_mean.shape}")
print(f"n_observations: {all_obs.shape[0]}")
print(f"n_images: {n_images}")
print(f"Point source mags (H5, this epoch): {point_mags_h5}")
print(f"Point source mean flux (H5, cps):   {[f'{f:.1f}' for f in point_mean_flux_h5]}")
print(f"Point dx (pix): {point_dx}")
print(f"Point dy (pix): {point_dy}")

## H5 difference + rotation

In [ ]:
# H5 difference: epoch stamp minus mean stamp (analogous to visit - template)
h5_diff = h5_stamp - h5_stamp_mean

# Rotate stamp to match visit orientation
# LSST pipeline uses makeRotation(angle) which is CCW in (x,y) pixel space
# scipy_rotate(arr, angle) is CCW in (row,col) array space, equivalent to CW in display
# These are the same transformation, so use positive angle (NOT negative)
h5_stamp_rot = scipy_rotate(h5_stamp, visit_rotation_angle,
                            reshape=True, order=4, mode='constant', cval=0.0)
h5_diff_rot = scipy_rotate(h5_diff, visit_rotation_angle,
                           reshape=True, order=4, mode='constant', cval=0.0)

print(f"Rotated stamp shape: {h5_stamp_rot.shape}")
print(f"Rotated diff shape: {h5_diff_rot.shape}")

# Compute rotated point source positions
# Adjust for center shift due to reshape
new_center_x = (h5_diff_rot.shape[1] - 1) / 2.0
new_center_y = (h5_diff_rot.shape[0] - 1) / 2.0
print(f"new_center_x: {new_center_x}")
print(f"new_center_y: {new_center_y}")

offset_x = new_center_x - STAMP_CENTER
offset_y = new_center_y - STAMP_CENTER
print(f"offset_x: {offset_x}")
print(f"offset_y: {offset_y}")

rot_xs, rot_ys = [], []
for dx, dy in zip(point_dx, point_dy):
    x, y = compute_coord_after_rot(STAMP_CENTER, STAMP_CENTER, dx, dy, visit_rotation_angle)
    rot_xs.append(x + offset_x)
    rot_ys.append(y + offset_y)

print(f"Rotated point positions (x): {[f'{x:.1f}' for x in rot_xs]}")
print(f"Rotated point positions (y): {[f'{y:.1f}' for y in rot_ys]}")

## Side-by-side comparison

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 5))

# Panel 1: Pipeline visit cutout
im0 = axes[0].imshow(np.arcsinh(cutout_visit), origin='lower', cmap='gray')
axes[0].set_title('Pipeline: Visit Cutout')
plt.colorbar(im0, ax=axes[0], fraction=0.046)
cx0 = (cutout_visit.shape[1] - 1) / 2.0
cy0 = (cutout_visit.shape[0] - 1) / 2.0
axes[0].plot(cx0, cy0, 'cx', ms=12, mew=2)

# Panel 2: Pipeline difference cutout
data2 = np.arcsinh(cutout_diff)
vlim2 = max(abs(np.nanmin(data2)), abs(np.nanmax(data2)))
im1 = axes[1].imshow(data2, origin='lower', cmap='bwr', vmin=-vlim2, vmax=vlim2)
axes[1].set_title('Pipeline: Diff Cutout')
plt.colorbar(im1, ax=axes[1], fraction=0.046)
cx1 = (cutout_diff.shape[1] - 1) / 2.0
cy1 = (cutout_diff.shape[0] - 1) / 2.0
axes[1].plot(cx1, cy1, 'cx', ms=12, mew=2)

# Panel 3: Rotated H5 stamp
im2 = axes[2].imshow(np.arcsinh(h5_stamp_rot), origin='lower', cmap='gray')
axes[2].set_title('H5: Rotated Stamp')
plt.colorbar(im2, ax=axes[2], fraction=0.046)
axes[2].plot(new_center_x, new_center_y, 'cx', ms=12, mew=2)
for i, (x, y) in enumerate(zip(rot_xs, rot_ys)):
    axes[2].plot(x, y, 'm+', ms=10, mew=2)
    axes[2].text(x + 0.5, y + 0.5, f'img_{i}', color='m', fontsize=7)

# Panel 4: Rotated H5 difference with Gaussian smoothing
h5_diff_rot_smooth = gaussian_filter(h5_diff_rot, sigma=PSF_SIGMA)
data4 = np.arcsinh(h5_diff_rot_smooth)
vlim4 = max(abs(np.nanmin(data4)), abs(np.nanmax(data4)))
im3 = axes[3].imshow(data4, origin='lower', cmap='bwr', vmin=-vlim4, vmax=vlim4)
axes[3].set_title(f'H5: Rotated Diff (smoothed, $\\sigma$={PSF_SIGMA:.1f})')
plt.colorbar(im3, ax=axes[3], fraction=0.046)
axes[3].plot(new_center_x, new_center_y, 'cx', ms=12, mew=2)
for i, (x, y) in enumerate(zip(rot_xs, rot_ys)):
    axes[3].plot(x, y, 'm+', ms=10, mew=2)
    axes[3].text(x + 0.5, y + 0.5, f'img_{i}', color='m', fontsize=7)

plt.suptitle(f'System {system_index}, band={band}, time_index={time_index}, '
             f'rot={visit_rotation_angle:.1f} deg', fontsize=14)
plt.tight_layout()
plt.show()

## Point source overlay on pipeline diff cutout

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
data_overlay = np.arcsinh(cutout_diff)
vlim_overlay = max(abs(np.nanmin(data_overlay)), abs(np.nanmax(data_overlay)))
im = ax.imshow(data_overlay, origin='lower', cmap='bwr', vmin=-vlim_overlay, vmax=vlim_overlay)
plt.colorbar(im, ax=ax)

cx = (cutout_diff.shape[1] - 1) / 2.0
cy = (cutout_diff.shape[0] - 1) / 2.0
ax.plot(cx, cy, 'cx', ms=12, mew=2)

cutout_wcs = astropyWCS(cutout_diff_header)
for i, p in enumerate(associated_points):
    px, py = cutout_wcs.world_to_pixel_values(p['ra'], p['dec'])
    ax.plot(px, py, 'm+', ms=12, mew=2)
    ax.text(px + 1, py + 1, f'img_{i}\nmag={p["mag"]:.1f}', color='m', fontsize=8)

ax.set_title(f'Diff cutout with point sources\nSystem {system_index}')
plt.tight_layout()
plt.show()

## Flux comparison (difference image)

The difference image shows flux_visit - flux_template ≈ flux_epoch - flux_mean.
Convert H5 mags to fluxes and compare the difference flux against the cutout aperture flux.

In [ ]:
cutout_wcs = astropyWCS(cutout_diff_header)

# H5 mags use CPS zero points → convert to counts/sec
# Cutout diff is likely in counts/sec (or nJy); compare shape, not absolute values
print(f"{'Image':>8} {'Epoch mag':>10} {'Epoch flux':>12} {'Mean flux':>12} {'H5 Δflux':>12} {'Cutout Δflux':>12}")
print('-' * 72)
for i, p in enumerate(associated_points):
    px, py = cutout_wcs.world_to_pixel_values(p['ra'], p['dec'])
    px, py = int(np.round(px)), int(np.round(py))

    flux_epoch = mag2flux_cps(point_mags_h5[i], band)
    flux_mean = point_mean_flux_h5[i]
    h5_delta_flux = flux_epoch - flux_mean

    if 0 <= px < cutout_diff.shape[1] and 0 <= py < cutout_diff.shape[0]:
        r = 2
        aperture = cutout_diff[max(0, py-r):py+r+1, max(0, px-r):px+r+1]
        cutout_flux = np.sum(aperture)
        print(f"  img_{i}: {point_mags_h5[i]:10.2f} {flux_epoch:12.1f} {flux_mean:12.1f} "
              f"{h5_delta_flux:12.1f} {cutout_flux:12.1f}")
    else:
        print(f"  img_{i}: {point_mags_h5[i]:10.2f} {flux_epoch:12.1f} {flux_mean:12.1f} "
              f"{h5_delta_flux:12.1f}  (outside cutout)")